In [1]:
# Exercise 1: Pretrained sentiment analysis
# Sources: Lab 3/docs/08_implementation.md; Lab 3/docs/03_model.md; Lab 3/docs/07_demo.md
import torch
from transformers import pipeline

classifier = pipeline('text-classification', model='tabularisai/multilingual-sentiment-analysis')
text = 'Bộ phim này rất hay!'
print('Tokens:', classifier.tokenizer.tokenize(text))
print(classifier(text, truncation=True, max_length=512))
del classifier


C:\Users\Legion\Desktop\TÀI LIỆU HỌC\Lab\Lab 3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device set to use cuda:0


Tokens: ['Bộ', 'phim', 'này', 'rất', 'hay', '!']
[{'label': 'Positive', 'score': 0.6520066261291504}]


In [2]:
# Exercise 2 - Step 1: Prepare libraries
# Sources: Lab 3/docs/08_implementation.md; Lab 3/docs/10_sources.md
from pathlib import Path
from datetime import datetime
import json
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, ConfusionMatrixDisplay
from transformers import (pipeline, AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer, TrainerCallback, EarlyStoppingCallback, set_seed)

ROOT = Path('Lab 3') if Path('Lab 3').is_dir() else Path('.')
MODEL_NAME = 'distilbert/distilbert-base-uncased'
set_seed(42)
print('Thiết bị:', 'CUDA' if torch.cuda.is_available() else 'CPU')


Thiết bị: CUDA


In [3]:
# Exercise 2 - Step 2: Load and split IMDB
# Sources: Lab 3/docs/02_data.md; Lab 3/docs/08_implementation.md
raw_train, raw_test = load_dataset('stanfordnlp/imdb', split=['train', 'test'])
train_ids, val_ids = train_test_split(np.arange(len(raw_train)), test_size=0.2,
                                     stratify=raw_train['label'], random_state=42)
data = DatasetDict(train=raw_train.select(train_ids),
                   validation=raw_train.select(val_ids), test=raw_test)
print({name: len(split) for name, split in data.items()})


{'train': 20000, 'validation': 5000, 'test': 25000}


In [4]:
# Exercise 2 - Step 3: Load pretrained model and tokenizer
# Sources: Lab 3/docs/03_model.md; Lab 3/docs/04_training.md; Lab 3/docs/08_implementation.md
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

CHECKPOINT = ROOT/'outputs/full/checkpoints/best'
TRAIN = not (CHECKPOINT/'config.json').exists()
OUTPUT = ROOT/'outputs/simple-training'
if TRAIN:
    OUTPUT = OUTPUT/datetime.now().strftime('%Y%m%d-%H%M%S')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME if TRAIN else str(CHECKPOINT), num_labels=2,
    id2label={0: 'NEGATIVE', 1: 'POSITIVE'}, label2id={'NEGATIVE': 0, 'POSITIVE': 1})
print('Chế độ:', 'Huấn luyện mới' if TRAIN else 'Nạp model đã huấn luyện')


Chế độ: Nạp model đã huấn luyện


In [5]:
# Exercise 2 - Step 4: Tokenize datasets
# Sources: Lab 3/docs/02_data.md; Lab 3/docs/03_model.md
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=256)

tokenized = data.map(tokenize, batched=True, remove_columns=['text'])
collator = DataCollatorWithPadding(tokenizer)


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   4%|▍         | 1000/25000 [00:00<00:03, 6359.55 examples/s]

Map:   8%|▊         | 2000/25000 [00:00<00:05, 4316.33 examples/s]

Map:  12%|█▏        | 3000/25000 [00:00<00:04, 5344.36 examples/s]

Map:  16%|█▌        | 4000/25000 [00:00<00:03, 5737.93 examples/s]

Map:  20%|██        | 5000/25000 [00:00<00:03, 6186.24 examples/s]

... (17 outputs left out)

Map:  92%|█████████▏| 23000/25000 [00:03<00:00, 5865.37 examples/s]

Map:  96%|█████████▌| 24000/25000 [00:03<00:00, 5987.83 examples/s]

Map: 100%|██████████| 25000/25000 [00:04<00:00, 6121.32 examples/s]

Map: 100%|██████████| 25000/25000 [00:04<00:00, 6168.13 examples/s]

In [6]:
# Exercise 2 - Step 5: Configure training arguments
# Sources: Lab 3/docs/04_training.md; Lab 3/docs/08_implementation.md
args = TrainingArguments(
    output_dir=str(OUTPUT), eval_strategy='epoch', save_strategy='epoch',
    learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
    num_train_epochs=3, weight_decay=0.01, max_grad_norm=1.0, optim='adamw_torch',
    load_best_model_at_end=True, metric_for_best_model='accuracy', greater_is_better=True,
    save_total_limit=2, fp16=torch.cuda.is_available(), report_to='none',
    seed=42, logging_steps=50)


In [ ]:
# Exercise 2 - Step 6: Define metrics and train
# Sources: Lab 3/docs/05_evaluation.md; Lab 3/docs/04_training.md; Lab 3/docs/08_implementation.md; Lab 3/docs/11_AI_Usage.md
def compute_metrics(prediction):
    logits, labels = prediction
    predicted = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predicted, average='binary', zero_division=0)
    return {'accuracy': accuracy_score(labels, predicted), 'precision': precision,
            'recall': recall, 'f1_binary': f1,
            'f1_macro': f1_score(labels, predicted, average='macro', zero_division=0)}

def save_training_time(path, values):
    values = {'sources_md': ['docs/04_training.md', 'docs/08_implementation.md'], **values}
    path.write_text(json.dumps(values, ensure_ascii=False, indent=2), encoding='utf-8')

class TrainingTime(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.started_at = datetime.now().astimezone().isoformat()
        self.start = time.perf_counter()
        self.epochs = []

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.epoch_start = time.perf_counter()

    def on_save(self, args, state, control, **kwargs):
        row = {'epoch': state.epoch, 'step': state.global_step,
               'saved_at': datetime.now().astimezone().isoformat(),
               'epoch_seconds': round(time.perf_counter() - self.epoch_start, 3),
               'elapsed_train_seconds': round(time.perf_counter() - self.start, 3)}
        self.epochs.append(row)
        path = Path(args.output_dir)/f'checkpoint-{state.global_step}'/'training_time.json'
        save_training_time(path, {'scope': 'checkpoint', 'started_at': self.started_at, **row})

timing = TrainingTime()
trainer = Trainer(
    model=model, args=args, train_dataset=tokenized['train'], eval_dataset=tokenized['validation'],
    processing_class=tokenizer, data_collator=collator, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.01), timing])

if TRAIN:
    (OUTPUT/'SOURCES.md').write_text(
        '# Nguồn và phạm vi\n\nÁp dụng mọi artifact trong run này.\n'
        'Căn cứ (từ Lab 3/): docs/08_implementation.md §6; docs/04_training.md §6; docs/03_model.md; docs/02_data.md.\n',
        encoding='utf-8')
    started_at, start = datetime.now().astimezone().isoformat(), time.perf_counter()
    status = 'interrupted_or_failed'
    try:
        trainer.train()
        status = 'completed'
    finally:
        training_time = {
            'scope': 'run', 'status': status,
            'started_at': getattr(timing, 'started_at', started_at),
            'ended_at': datetime.now().astimezone().isoformat(),
            'total_train_seconds': round(time.perf_counter() - getattr(timing, 'start', start), 3),
            'best_model_checkpoint': trainer.state.best_model_checkpoint,
            'epochs': getattr(timing, 'epochs', [])}
        save_training_time(OUTPUT/'training_time.json', training_time)
    CHECKPOINT = OUTPUT/'best'
    trainer.save_model(str(CHECKPOINT))
    tokenizer.save_pretrained(str(CHECKPOINT))
    trainer.save_state()
    save_training_time(CHECKPOINT/'training_time.json', training_time)
    print('Thời gian train:', training_time['total_train_seconds'], 'giây')
else:
    print('Đã nạp checkpoint; không thực hiện lượt huấn luyện mới.')
    timing_path = CHECKPOINT/'training_time.json'
    if timing_path.exists():
        print('Thời gian train đã lưu:', json.loads(timing_path.read_text(encoding='utf-8'))['total_train_seconds'], 'giây')


In [ ]:
# Exercise 2 - Step 7: Evaluate and demonstrate
# Sources: Lab 3/docs/05_evaluation.md; Lab 3/docs/06_verification.md; Lab 3/docs/07_demo.md; Lab 3/docs/03_model.md
result = trainer.predict(tokenized['test'])
print(result.metrics)
y_pred = result.predictions.argmax(axis=-1)
ConfusionMatrixDisplay.from_predictions(result.label_ids, y_pred, labels=[0, 1], display_labels=['NEGATIVE', 'POSITIVE'], cmap='Blues')
plt.show()

history_path = CHECKPOINT.parent/'trainer_state.json'
full_history_path = ROOT/'outputs/full/trainer_history.json'
if TRAIN:
    history = trainer.state.log_history
elif history_path.exists():
    history = json.loads(history_path.read_text(encoding='utf-8'))['log_history']
elif full_history_path.exists():
    history = json.loads(full_history_path.read_text(encoding='utf-8'))['history']
else:
    history = []
if history:
    train_log = [row for row in history if 'loss' in row]
    val_log = [row for row in history if 'eval_loss' in row]
    plt.plot([row['step'] for row in train_log], [row['loss'] for row in train_log], label='Train')
    plt.plot([row['step'] for row in val_log], [row['eval_loss'] for row in val_log], 'o-', label='Validation')
    plt.xlabel('Optimizer step'); plt.ylabel('Loss'); plt.legend(); plt.grid(alpha=0.3)
    plt.show()
else:
    print('Không có log loss của lượt train cũ để vẽ lại biểu đồ.')

sentiment = pipeline('text-classification', model=str(CHECKPOINT), tokenizer=str(CHECKPOINT),
                     device=0 if torch.cuda.is_available() else -1)
sentences = ['I loved this movie. The acting was great!', 'The movie was boring and disappointing.']
for sentence, prediction in zip(sentences, sentiment(sentences, truncation=True, max_length=256)):
    print(sentence, '->', prediction)
